# 사고 빈도(freMTPL2freq) 회귀 모델링

| 순서 | 내용 |
| --- | --- |
| 0 | 전처리 (기존 `사고빈도_전처리_분할없음.ipynb` 코드 그대로, 시각화 제외) |
| 1 | 적절한 피처에 스케일링 |
| 2 | X, y 변수를 새 DataFrame으로 추출 |
| 3 | train_test_split으로 데이터 분리 |
| 4 | fit() · predict()로 예측 (LinearRegression, PoissonRegressor) |
| 5 | KFold 교차 검증 (RMSE · R² · 포아송 deviance) |
| 6 | GridSearchCV로 스케일러 · 파라미터 탐색 |
| 7 | 점수 비교표: 기본 모델 → 교차 검증 평균 → 튜닝 후 |

## 0. 전처리 (기존 코드 그대로)

In [1]:
import numpy as np
import pandas as pd

In [2]:
# ── 1단계: 데이터 불러오기 ──
freq = pd.read_csv("freMTPL2freq.csv")
print(f"계약 수: {len(freq):,}")
print(f"총 사고 건수: {freq.ClaimNb.sum():,}")
print(f"총 가입 기간(년): {freq.Exposure.sum():,.0f}")
print(f"단순 계산 1년당 사고율 = 총 사고 ÷ 총 기간 = "
      f"{freq.ClaimNb.sum() / freq.Exposure.sum():.4f}")

계약 수: 678,013
총 사고 건수: 36,102
총 가입 기간(년): 358,499
단순 계산 1년당 사고율 = 총 사고 ÷ 총 기간 = 0.1007


In [3]:
# ---------------------------------------------------------------- 1. 타깃 ClaimNb 점검
# 사고 5건 이상 9건 중 7건은 같은 차가 여러 계약 번호로 쪼개 기록된 것으로 보임
# 행을 지우면 "위험하다"는 정보까지 사라지므로, 삭제하지 않고 4로 cap
n_over = (freq.ClaimNb > 4).sum()
freq["ClaimNb"] = freq["ClaimNb"].clip(upper=4)
print(f"[1] ClaimNb 4에서 cap: {n_over}건 수정")

[1] ClaimNb 4에서 cap: 9건 수정


In [4]:
# ---------------------------------------------------------------- 2. Exposure 점검
# 1년 넘는 계약(대부분 1.01~1.04)은 기록 오차 → 1로 cap
# 분모가 커지면 1년당 사고율이 작게 계산됨 (예: 1 ÷ 2.01 = 0.50)
# 짧은 계약은 진짜 값이므로 그대로 둠 → offset이 알아서 보정
n_over = (freq.Exposure > 1).sum()
freq["Exposure"] = freq["Exposure"].clip(upper=1)
print(f"[2] Exposure 1에서 cap: {n_over:,}건 수정")

[2] Exposure 1에서 cap: 1,224건 수정


In [5]:
# ---------------------------------------------------------------- 3. 열 고르기
# ClaimNb = 타깃, Exposure = offset (심도와 반대로 버리지 않음!)
# Area는 Density를 6구간으로 자른 것 → 둘 다 넣으면 배수가 쪼개지고 SE가 커짐
freq = freq.drop(columns=["Area"])
print("[3] Area 제거, Exposure는 offset용으로 유지")

[3] Area 제거, Exposure는 offset용으로 유지


In [6]:
# ---------------------------------------------------------------- 4. 변수 변환 (전체 데이터에 바로 적용)
# 분할을 안 하므로 train에서 기준을 "배울" 필요 없이 전체 데이터로 바로 판단
# 희귀 범주는 행 수가 아니라 "사고 수"로 판단
# (사고 45건인 지역은 5건만 더 나도 사고율이 11% 흔들림)
def frequent_levels(d, col, min_claims=200):
    claims = d.groupby(col)["ClaimNb"].sum()
    return set(claims[claims >= min_claims].index)


keep_region = frequent_levels(freq, "Region")
keep_brand = frequent_levels(freq, "VehBrand")
print(f"[4] Other로 합칠 Region: {sorted(set(freq.Region) - keep_region)}")
print(f"    Other로 합칠 VehBrand: {sorted(set(freq.VehBrand) - keep_brand)}")

AGE_BINS = [17, 21, 25, 30, 40, 50, 60, 70, 80, 101]

freq["VehAge"] = freq["VehAge"].clip(upper=20)
freq["BonusMalus"] = freq["BonusMalus"].clip(upper=150)
freq["DrivAgeBin"] = pd.cut(freq["DrivAge"], bins=AGE_BINS).astype(str)
freq["VehPower"] = freq["VehPower"].clip(upper=12).astype(str)
freq["logDensity"] = np.log(freq["Density"])
freq["Region"] = freq["Region"].where(freq["Region"].isin(keep_region), "Other")
freq["VehBrand"] = freq["VehBrand"].where(freq["VehBrand"].isin(keep_brand), "Other")

[4] Other로 합칠 Region: ['Alsace', 'Auvergne', 'Champagne-Ardenne', 'Franche-Comte']


    Other로 합칠 VehBrand: ['B14']


In [7]:
# ---------------------------------------------------------------- 5. 최종 점검 + 저장
# 실수가 있어도 모델은 조용히 돌아가므로, 규칙이 지켜졌는지 assert로 바로 멈춰서 확인
cols = ["ClaimNb", "Exposure", "DrivAgeBin", "VehAge", "BonusMalus", "VehPower",
        "VehGas", "VehBrand", "Region", "logDensity"]
assert freq["IDpol"].is_unique, "IDpol 중복"
assert freq[cols].isna().sum().sum() == 0, "결측치 있음"
assert freq["ClaimNb"].max() <= 4, "ClaimNb cap 누락"
# Exposure = 0이면 offset log(0) = -inf → numpy는 경고만 내고 계속 계산함
assert freq["Exposure"].between(0, 1, inclusive="right").all(), "Exposure가 0 이하 또는 1 초과"
assert "nan" not in set(freq["DrivAgeBin"]), "나이 구간에 안 들어간 운전자 있음"
print(f"[5] 점검 통과 ✅  계약 {len(freq):,}건 · "
      f"1년당 사고율 {freq.ClaimNb.sum() / freq.Exposure.sum():.4f}")

keep = ["IDpol"] + cols
freq[keep].to_csv("freq_prep.csv", index=False)
print("\n저장: freq_prep.csv")

[5] 점검 통과 ✅  계약 678,013건 · 1년당 사고율 0.1006



저장: freq_prep.csv


## 모델링
- **타깃 y** = `ClaimNb / Exposure` (1년당 사고율). 가입 기간이 계약마다 달라서 건수 대신 사고율을 맞힌다.
- **가중치** = `Exposure`. 오래 가입한 계약일수록 사고율을 더 믿을 수 있으므로 fit과 점수 계산 모두 `Exposure`로 가중한다.
  (offset `log(Exposure)`를 직접 넣을 수 없는 scikit-learn에서 쓰는 표준 방법, 포아송 모델에서는 offset과 같은 결과)

In [8]:
import warnings

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.metrics import mean_poisson_deviance, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. 스케일링할 피처 고르기

In [9]:
# ---------------------------------------------------------------- 1. 스케일링 대상 확인
# 숫자 열만 스케일링 후보. 범주 열은 원-핫(0/1)으로 바꾸므로 스케일링하지 않음
NUM_COLS = ["VehAge", "BonusMalus", "logDensity"]                     # 스케일링 O
CAT_COLS = ["DrivAgeBin", "VehPower", "VehGas", "VehBrand", "Region"]  # 원-핫 인코딩
# ClaimNb(타깃), Exposure(가중치), IDpol(확인용)은 설명변수가 아니라서 스케일링 X

# VehPower는 전처리에서 문자열로 바꿨지만, csv를 다시 읽는 경우를 대비해 한 번 더 문자열로
freq["VehPower"] = freq["VehPower"].astype(str)

check = freq[NUM_COLS].agg(["min", "max", "mean", "std", "skew"]).T
check["범위(max-min)"] = check["max"] - check["min"]
print("[1] 숫자 피처의 크기(단위)가 서로 다름 → 스케일링 필요")
display(check)

[1] 숫자 피처의 크기(단위)가 서로 다름 → 스케일링 필요


,min,max,mean,std,skew,범위(max-min)
VehAge,0.0000,20.0000,6.9761,5.3990,0.5210,20.0000
BonusMalus,50.0000,150.0000,59.7572,15.6079,1.7015,100.0000
logDensity,0.0000,10.2036,5.9818,1.8705,0.0286,10.2036


- 세 숫자 피처의 범위가 20, 100, 10으로 서로 다르다. 계수에 벌점(규제)을 주는 모델(PoissonRegressor의 `alpha`)은 피처 크기가 다르면 벌점이 불공평하게 걸린다.
- 이미 전처리에서 상한(cap)·log 변환을 해서 극단값이 없으므로 표준화(StandardScaler)와 정규화(MinMaxScaler)를 둘 다 쓸 수 있다. 어느 쪽이 나은지는 6단계 GridSearchCV에서 비교한다.
- 스케일러는 **train에서만 fit**해야 test 정보가 새지 않는다. 그래서 스케일러를 `Pipeline` 안에 넣어 분할 뒤 학습 데이터로만 맞추게 한다.

In [10]:
def make_pipeline(model, scaler=None):
    """숫자 열은 스케일링, 범주 열은 원-핫 인코딩한 뒤 모델에 넣는 파이프라인
    Exposure 등 나머지 열은 remainder='drop'으로 모델에 들어가지 않음"""
    prep = ColumnTransformer([
        ("num", scaler if scaler is not None else StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT_COLS),
    ], remainder="drop")
    return Pipeline([("prep", prep), ("model", model)])

## 2. X, y를 새 DataFrame으로 추출

In [11]:
# ---------------------------------------------------------------- 2. X, y 추출
# X에 Exposure를 함께 두는 이유: 분할·교차 검증 때 행이 같이 나뉘어야 가중치로 쓸 수 있음
# (모델에는 들어가지 않음 → make_pipeline의 remainder="drop")
X = freq[NUM_COLS + CAT_COLS + ["Exposure"]].copy()
y = freq["ClaimNb"] / freq["Exposure"]   # 1년당 사고율
y.name = "Frequency"

print(f"[2] X {X.shape} · y {y.shape}")
print(f"    y(1년당 사고율) 가중 평균 = {np.average(y, weights=X['Exposure']):.4f}")
display(X.head())

[2] X (678013, 9) · y (678013,)
    y(1년당 사고율) 가중 평균 = 0.1006


,VehAge,BonusMalus,logDensity,DrivAgeBin,VehPower,VehGas,VehBrand,Region,Exposure
0,0,50,7.1041,"(50, 60]",5,Regular,B12,Rhone-Alpes,0.1000
1,0,50,7.1041,"(50, 60]",5,Regular,B12,Rhone-Alpes,0.7700
2,2,50,3.9890,"(50, 60]",6,Diesel,B12,Picardie,0.7500
3,0,50,4.3307,"(40, 50]",7,Diesel,B12,Aquitaine,0.0900
4,0,50,4.3307,"(40, 50]",7,Diesel,B12,Aquitaine,0.8400


## 3. train_test_split으로 분리

In [12]:
# ---------------------------------------------------------------- 3. 분할
# 사고 0건이 95% → 드문 3·4건 계약이 한쪽에 몰리지 않게 ClaimNb로 층화, 8:2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=freq["ClaimNb"])
w_train, w_test = X_train["Exposure"], X_test["Exposure"]

print(f"[3] train {X_train.shape} · test {X_test.shape}")
for name, yy, ww in [("train", y_train, w_train), ("test", y_test, w_test)]:
    print(f"    {name} 1년당 사고율 = {np.average(yy, weights=ww):.4f}")

# 스케일링이 train 기준으로 잘 되었는지 확인 (StandardScaler → 평균 0, 표준편차 1)
scaler_check = StandardScaler().fit(X_train[NUM_COLS])
scaled = pd.DataFrame(scaler_check.transform(X_train[NUM_COLS]), columns=NUM_COLS)
print("\n[1] 스케일링 전후 (train)")
display(pd.concat({"전": X_train[NUM_COLS].agg(["mean", "std"]),
                   "후 (StandardScaler)": scaled.agg(["mean", "std"])}, axis=1))

[3] train (542410, 9) · test (135603, 9)
    train 1년당 사고율 = 0.1006
    test 1년당 사고율 = 0.1007

[1] 스케일링 전후 (train)


전                       후 (StandardScaler)                      
     VehAge BonusMalus logDensity             VehAge BonusMalus logDensity
mean 6.9826    59.7519     5.9819            -0.0000     0.0000    -0.0000
std  5.3981    15.5967     1.8706             1.0000     1.0000     1.0000

## 4. fit() · predict()
평가 지표 (모두 `Exposure`로 가중)
- **RMSE · R²**: 과제에서 제시한 일반 회귀 지표
- **평균 포아송 deviance**: 팀이 정한 주 지표 (작을수록 좋음). 개수 자료에 맞는 오차
- **총 건수 비율**: 예측 사고 수 합 ÷ 실제 사고 수 합 (1에 가까울수록 좋음)
- **lift**: 예측 상위 20% 계약의 실제 사고율 ÷ 하위 20% 계약의 실제 사고율 (클수록 위험한 계약을 잘 구분)

In [13]:
# ---------------------------------------------------------------- 4. 평가 함수
def evaluate(y_true, pred, w):
    """y_true, pred = 1년당 사고율, w = Exposure"""
    # LinearRegression은 음수를 예측할 수 있음 → 포아송 deviance는 양수만 받으므로 아주 작은 값으로 올림
    pos = np.clip(pred, 1e-6, None)
    y_true, w = np.asarray(y_true), np.asarray(w)
    # lift: 예측 사고율 순으로 계약을 5그룹으로 나눠 그룹별 실제 1년당 사고율 비교
    group = pd.qcut(pd.Series(pred).rank(method="first"), 5, labels=False).to_numpy()
    rate = pd.Series(y_true * w).groupby(group).sum() / pd.Series(w).groupby(group).sum()
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, pred, sample_weight=w)),
        "R2": r2_score(y_true, pred, sample_weight=w),
        "포아송 deviance": mean_poisson_deviance(y_true, pos, sample_weight=w),
        "총 건수 비율": (pred * w).sum() / (y_true * w).sum(),
        "lift": rate[4] / rate[0],
        "음수 예측 수": int((pred <= 0).sum()),
    }


# 교차 검증 · GridSearchCV용 점수 함수 (scikit-learn은 '클수록 좋음'이라 오차는 음수로)
# 검증 폴드의 X에 들어 있는 Exposure를 꺼내 가중치로 씀
def _scorer(key, sign):
    def score(estimator, X_, y_):
        return sign * evaluate(y_, estimator.predict(X_), X_["Exposure"])[key]
    return score


SCORING = {"deviance": _scorer("포아송 deviance", -1),
           "rmse": _scorer("RMSE", -1),
           "r2": _scorer("R2", 1)}

In [14]:
# ---------------------------------------------------------------- 4. 기본 모델 학습 · 예측
# 기준선: 모든 계약에 train 평균 사고율을 예측 (모델이 이것보다 나아야 의미 있음)
base_rate = np.average(y_train, weights=w_train)

models = {
    "LinearRegression": make_pipeline(LinearRegression()),
    # PoissonRegressor 기본값 alpha=1 (규제 세기) 그대로 사용 → 6단계에서 튜닝
    "PoissonRegressor": make_pipeline(PoissonRegressor(max_iter=1000)),
}

test_scores = {"기준선(평균)": evaluate(y_test, np.full(len(y_test), base_rate), w_test)}
for name, pipe in models.items():
    pipe.fit(X_train, y_train, model__sample_weight=w_train)   # 가중치는 모델 단계로 전달
    test_scores[name] = evaluate(y_test, pipe.predict(X_test), w_test)

basic = pd.DataFrame(test_scores).T
print("[4] 기본 모델 test 점수")
display(basic)

[4] 기본 모델 test 점수


,RMSE,R2,포아송 deviance,총 건수 비율,lift,음수 예측 수
기준선(평균),0.7204,-0.0000,0.6230,0.9984,1.0150,0.0000
LinearRegression,0.7185,0.0053,0.6004,1.0003,3.9049,351.0000
PoissonRegressor,0.7201,0.0007,0.6194,0.9987,3.3092,0.0000


## 5. KFold 교차 검증 (train 안에서)

In [15]:
# ---------------------------------------------------------------- 5. 교차 검증
# test는 마지막 평가용으로 남겨 두고, train을 5조각으로 나눠 번갈아 검증
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_folds, cv_mean = {}, {}
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=kf, scoring=SCORING,
                         params={"model__sample_weight": w_train})
    folds = pd.DataFrame({"포아송 deviance": -res["test_deviance"],
                          "RMSE": -res["test_rmse"],
                          "R2": res["test_r2"]},
                         index=[f"Fold {i}" for i in range(1, 6)])
    cv_folds[name] = folds
    cv_mean[name] = folds.mean()
    print(f"[5] {name} — 폴드별 점수")
    display(pd.concat([folds, folds.agg(["mean", "std"]).rename(index={"mean": "평균", "std": "표준편차"})]))

[5] LinearRegression — 폴드별 점수


,포아송 deviance,RMSE,R2
Fold 1,0.6110,0.7387,0.0049
Fold 2,0.6013,0.7056,0.0046
Fold 3,0.6085,0.7711,0.0048
Fold 4,0.6096,0.7426,0.0040
Fold 5,0.6047,0.7353,0.0046
평균,0.6070,0.7387,0.0046
표준편차,0.0040,0.0233,0.0004


[5] PoissonRegressor — 폴드별 점수


,포아송 deviance,RMSE,R2
Fold 1,0.6278,0.7403,0.0007
Fold 2,0.6155,0.7070,0.0007
Fold 3,0.6265,0.7727,0.0006
Fold 4,0.6189,0.7439,0.0006
Fold 5,0.6207,0.7367,0.0006
평균,0.6219,0.7401,0.0006
표준편차,0.0052,0.0234,0.0000


## 6. GridSearchCV
- **LinearRegression**: 바꿀 하이퍼파라미터가 없어서 스케일러만 비교 (표준화 · 정규화 · 스케일링 안 함)
- **PoissonRegressor**: 규제 세기 `alpha` × 스케일러 조합을 비교
- 기준(refit)은 주 지표인 포아송 deviance

In [16]:
# ---------------------------------------------------------------- 6. GridSearchCV
SCALERS = [StandardScaler(), MinMaxScaler(), "passthrough"]   # passthrough = 스케일링 안 함

grids = {
    "LinearRegression": {"prep__num": SCALERS},
    "PoissonRegressor": {"prep__num": SCALERS,
                         "model__alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1.0]},
}

searches = {}
for name, grid in grids.items():
    gs = GridSearchCV(models[name], grid, cv=kf, scoring=SCORING, refit="deviance", n_jobs=-1)
    gs.fit(X_train, y_train, model__sample_weight=w_train)
    searches[name] = gs

    table = pd.DataFrame(gs.cv_results_["params"])
    table["prep__num"] = table["prep__num"].map(lambda s: type(s).__name__ if s != "passthrough" else "스케일링 안 함")
    table["CV 포아송 deviance"] = -gs.cv_results_["mean_test_deviance"]
    table["CV RMSE"] = -gs.cv_results_["mean_test_rmse"]
    table["CV R2"] = gs.cv_results_["mean_test_r2"]
    table["순위"] = gs.cv_results_["rank_test_deviance"]
    table["학습 시간(초)"] = gs.cv_results_["mean_fit_time"]
    table = table.rename(columns={"prep__num": "스케일러", "model__alpha": "alpha"})
    print(f"[6] {name} — best_params_ = {gs.best_params_}")
    display(table.sort_values("순위").set_index("순위").style.format(precision=5))

[6] LinearRegression — best_params_ = {'prep__num': StandardScaler()}


,스케일러,CV 포아송 deviance,CV RMSE,CV R2,학습 시간(초)
순위,,,,,
1,StandardScaler,0.60700,0.73866,0.00456,2.16921
2,MinMaxScaler,0.60700,0.73866,0.00456,2.39634
3,스케일링 안 함,0.60702,0.73866,0.00456,1.88518


[6] PoissonRegressor — best_params_ = {'model__alpha': 0.0001, 'prep__num': 'passthrough'}


,alpha,스케일러,CV 포아송 deviance,CV RMSE,CV R2,학습 시간(초)
순위,,,,,,
1,0.00010,스케일링 안 함,0.60410,0.73869,0.00449,14.46635
2,0.00100,스케일링 안 함,0.60410,0.73868,0.00450,12.25567
3,0.00010,MinMaxScaler,0.60411,0.73867,0.00454,1.35338
4,0.00010,StandardScaler,0.60411,0.73869,0.00447,1.65726
5,0.00100,StandardScaler,0.60412,0.73868,0.00451,1.81185
6,0.01000,스케일링 안 함,0.60482,0.73871,0.00443,5.47697
7,0.01000,StandardScaler,0.60501,0.73871,0.00444,1.97437
8,0.00100,MinMaxScaler,0.60539,0.73874,0.00434,1.73708
9,0.10000,스케일링 안 함,0.60661,0.73883,0.00410,2.12357


**해석 포인트**
- LinearRegression은 규제가 없어서 스케일러를 바꿔도 점수가 같다 (계수 크기만 바뀌고 예측은 동일).
- PoissonRegressor는 `alpha`가 클수록 계수를 0 쪽으로 세게 당긴다. 기본값 `alpha=1`은 이 데이터에 너무 세서 성능이 기준선 수준으로 떨어진다 → `alpha`를 아주 작게(1e-4) 해야 한다.
- `alpha`가 작으면 스케일러끼리 점수 차이가 0.00001 수준으로 사실상 같다. 하지만 **스케일링 안 함**은 학습 시간이 약 9배 길다 (최적화가 늦게 수렴).
- `alpha`가 클 때 '스케일링 안 함'이 더 나아 보이는 것은 BonusMalus처럼 값이 큰 피처의 계수가 작게 잡혀 벌점을 덜 받기 때문이다. 피처마다 벌점이 불공평하게 걸린다는 뜻이라 오히려 스케일링이 필요한 이유가 된다.
- 그래서 최종 모델은 점수가 사실상 같으면서 빠르고 계수 비교가 공평한 **StandardScaler + 그 안에서 가장 좋은 alpha**로 고른다.

In [17]:
# ---------------------------------------------------------------- 6. 최종 PoissonRegressor 선택
res = pd.DataFrame(searches["PoissonRegressor"].cv_results_)
is_std = res["param_prep__num"].map(lambda s: isinstance(s, StandardScaler))
best_alpha = res.loc[res.loc[is_std, "rank_test_deviance"].idxmin(), "param_model__alpha"]

final_models = {
    "LinearRegression": searches["LinearRegression"].best_estimator_,
    "PoissonRegressor": make_pipeline(PoissonRegressor(alpha=best_alpha, max_iter=1000), StandardScaler())
                        .fit(X_train, y_train, model__sample_weight=w_train),
}
gap = res.loc[is_std, "mean_test_deviance"].max() - searches["PoissonRegressor"].best_score_
print(f"[6] 최종 PoissonRegressor: StandardScaler · alpha={best_alpha}")
print(f"    GridSearchCV 1위와의 CV deviance 차이 = {-gap:.6f}")

[6] 최종 PoissonRegressor: StandardScaler · alpha=0.0001
    GridSearchCV 1위와의 CV deviance 차이 = 0.000007


## 7. 점수 비교표: 기본 모델 → 교차 검증 평균 → 튜닝 후

In [18]:
# ---------------------------------------------------------------- 7. 최종 비교
tuned_scores = {name: evaluate(y_test, pipe.predict(X_test), w_test)
                for name, pipe in final_models.items()}

rows = []
for name in models:
    for stage, s in [("① 기본 모델 (test)", test_scores[name]),
                     ("② 교차 검증 평균 (train 5-fold)", cv_mean[name]),
                     ("③ 튜닝 후 (test)", tuned_scores[name])]:
        rows.append({"모델": name, "단계": stage,
                     "포아송 deviance": s["포아송 deviance"], "RMSE": s["RMSE"], "R2": s["R2"],
                     "총 건수 비율": s.get("총 건수 비율", np.nan), "lift": s.get("lift", np.nan)})
compare = pd.DataFrame(rows).set_index(["모델", "단계"])
print(f"[7] 기준선(모두 평균 {base_rate:.4f}으로 예측) test 포아송 deviance = "
      f"{test_scores['기준선(평균)']['포아송 deviance']:.4f}")
display(compare)

base = test_scores["기준선(평균)"]["포아송 deviance"]
for name in models:
    before, after = test_scores[name]["포아송 deviance"], tuned_scores[name]["포아송 deviance"]
    print(f"{name}: deviance {before:.4f} → {after:.4f} · 기준선 대비 개선 {(base - after) / base:.2%}"
          f" · 음수 예측 {tuned_scores[name]['음수 예측 수']}건")

[7] 기준선(모두 평균 0.1006으로 예측) test 포아송 deviance = 0.6230


포아송 deviance   RMSE     R2  \
모델               단계                                                      
LinearRegression ① 기본 모델 (test)                   0.6004 0.7185 0.0053   
                 ② 교차 검증 평균 (train 5-fold)        0.6070 0.7387 0.0046   
                 ③ 튜닝 후 (test)                    0.6004 0.7185 0.0053   
PoissonRegressor ① 기본 모델 (test)                   0.6194 0.7201 0.0007   
                 ② 교차 검증 평균 (train 5-fold)        0.6219 0.7401 0.0006   
                 ③ 튜닝 후 (test)                    0.5999 0.7185 0.0051   

                                            총 건수 비율   lift  
모델               단계                                         
LinearRegression ① 기본 모델 (test)              1.0003 3.9049  
                 ② 교차 검증 평균 (train 5-fold)      NaN    NaN  
                 ③ 튜닝 후 (test)               1.0003 3.9049  
PoissonRegressor ① 기본 모델 (test)              0.9987 3.3092  
                 ② 교차 검증 평균 (train 5-fold)      NaN    NaN  
                 ③ 튜닝 후 (test)               1.0006 3.8912

LinearRegression: deviance 0.6004 → 0.6004 · 기준선 대비 개선 3.62% · 음수 예측 351건
PoissonRegressor: deviance 0.6194 → 0.5999 · 기준선 대비 개선 3.72% · 음수 예측 0건


In [19]:
# ---------------------------------------------------------------- 7. 최종 모델의 주요 요인 exp(β)
# PoissonRegressor 계수 β → exp(β) = 사고율 배수 (1보다 크면 위험 ↑)
best = final_models["PoissonRegressor"]
names = best.named_steps["prep"].get_feature_names_out()
coef = pd.Series(np.exp(best.named_steps["model"].coef_), index=names, name="exp(β) 사고율 배수")

# 숫자 피처: StandardScaler라서 '1 표준편차 늘 때' 사고율 배수
num_coef = coef[[f"num__{c}" for c in NUM_COLS]].to_frame()
num_coef["1 표준편차"] = best.named_steps["prep"].named_transformers_["num"].scale_
print("[7] 숫자 피처: 1 표준편차 늘 때 사고율 배수")
display(num_coef)

# 범주 피처: 기준 범주(drop='first'로 빠진 첫 값) 대비 배수
ohe = best.named_steps["prep"].named_transformers_["cat"]
print("기준 범주:", {c: cats[0] for c, cats in zip(CAT_COLS, ohe.categories_)})
cat_coef = coef[coef.index.str.startswith("cat__")]
display(pd.concat([cat_coef.sort_values(ascending=False).head(5),
                   cat_coef.sort_values().head(5)]).to_frame())

[7] 숫자 피처: 1 표준편차 늘 때 사고율 배수


,exp(β) 사고율 배수,1 표준편차
num__VehAge,0.8022,5.3981
num__BonusMalus,1.4384,15.5967
num__logDensity,1.0866,1.8706


기준 범주: {'DrivAgeBin': '(17, 21]', 'VehPower': '10', 'VehGas': 'Diesel', 'VehBrand': 'B1', 'Region': 'Aquitaine'}


,exp(β) 사고율 배수
cat__VehBrand_B12,1.2102
cat__Region_Limousin,1.1481
cat__VehBrand_B11,1.1304
cat__Region_Bretagne,1.1244
cat__Region_Picardie,1.1237
"cat__DrivAgeBin_(25, 30]",0.7073
cat__VehPower_8,0.8058
"cat__DrivAgeBin_(21, 25]",0.8205
cat__VehPower_4,0.8233
"cat__DrivAgeBin_(30, 40]",0.8274
